In [13]:
import pandas as pd
import numpy as np

DATA_PATH = '../data/ready_to_use'


appointments = pd.read_csv(f'{DATA_PATH}/appointments_cleaned.csv')


In [14]:
# Fill missing reminder_channel values with the most frequent value (mode)
appointments.isna().sum()

appointments['reminder_channel'] = appointments['reminder_channel'].fillna(
    appointments['reminder_channel'].mode()[0],
)



In [15]:
# Fill missing reminder_days_before values with 0 when no reminder was sent (reminder_sent_flag = 0).

appointments['reminder_days_before'] = appointments[
    'reminder_days_before'
].fillna(appointments['reminder_sent_flag'].map({0: 0}))
appointments.isna().sum()


appointment_id                     0
patient_id                         0
provider_id                        0
facility_id                        0
appointment_date                   0
appointment_weekday                0
appointment_hour                   0
lead_days                          0
visit_type                         0
specialty_group                    0
payer_id                           0
reminder_channel                   0
reminder_sent_flag                 0
reminder_days_before               0
same_day_flag                      0
telehealth_flag                    0
distance_to_clinic_miles           0
transportation_barrier_flag        0
interpreter_needed_flag            0
weather_severity_group             0
zip_area_id                        0
patient_age                        0
sdoh_risk_score                    0
portal_active_flag                 0
deprivation_quintile               0
prior_completed_visits_12m         0
prior_no_show_count_12m            0
p

In [16]:
# Convert columns to datetime
appointments['appointment_date'] = pd.to_datetime(appointments['appointment_date'])
appointments['cancel_datetime'] = pd.to_datetime(appointments['cancel_datetime'])

# Calculate days before appointment

appointments['days_before_appointment'] = (
    appointments['appointment_date'].dt.normalize() -
    appointments['cancel_datetime'].dt.normalize()
).dt.days



In [17]:
appointments['cancel_flag'] = (
    appointments['days_before_appointment'].notna().astype(int)
)


In [18]:
appointments['appointment_month'] = appointments['appointment_date'].dt.month
appointments['appointment_day'] = appointments['appointment_date'].dt.day

In [19]:
appointments = appointments.drop(
    columns=['cancel_datetime','days_before_appointment'],
)


In [20]:
appointments["wait_to_room_minutes"] = appointments["wait_to_room_minutes"].fillna(-1)

In [21]:
categorical_cols = appointments.select_dtypes(include=['str', 'category']).columns

appointments[categorical_cols] = appointments[categorical_cols].apply(
    lambda col: col.str.strip()
)

In [22]:

weekday_map = {
    'Monday': 1,
    'Tuesday': 2,
    'Wednesday': 3,
    'Thursday': 4,
    'Friday': 5,
    'Saturday': 6,
    'Sunday': 7
}

appointments['appointment_weekday'] = (
    appointments['appointment_weekday'].str.strip().map(weekday_map)
)
appointments.head()

,appointment_id,patient_id,provider_id,facility_id,appointment_date,appointment_weekday,appointment_hour,lead_days,visit_type,specialty_group,...,portal_active_flag,deprivation_quintile,prior_completed_visits_12m,prior_no_show_count_12m,prior_cancel_count_12m,no_show_flag,wait_to_room_minutes,cancel_flag,appointment_month,appointment_day
0,APT000126,RBP011885,RBPV0067,SP03,2024-01-01,1,8,22,Behavioral Health,Behavioral health,...,1,4,0,0,0,0,22.0,0,1,1
1,APT010570,RBP005811,RBPV0014,PC03,2024-01-01,1,8,3,Acute,Primary care,...,1,2,0,0,0,0,6.0,0,1,1
2,APT010617,RBP010074,RBPV0105,UC02,2024-01-01,1,8,0,Urgent Care,Urgent care,...,1,2,0,0,0,0,7.0,0,1,1
3,APT027760,RBP011255,RBPV0025,PC03,2024-01-01,1,8,8,Follow-up,Primary care,...,1,3,0,0,0,0,-1.0,1,1,1
4,APT032568,RBP006231,RBPV0011,PC03,2024-01-01,1,9,2,Acute,Primary care,...,1,4,0,0,0,0,0.0,0,1,1


In [23]:
# check data info 
column_info = pd.DataFrame({
    'Column': appointments.columns,
    'Data Type': appointments.dtypes.values,
    'Non-Null Count': appointments.notnull().sum().values,
    'Null Count': appointments.isnull().sum().values,
    'Unique Values': appointments.nunique().values
})

display(column_info)

,Column,Data Type,Non-Null Count,Null Count,Unique Values
0,appointment_id,str,36000,0,36000
1,patient_id,str,36000,0,10512
2,provider_id,str,36000,0,110
3,facility_id,str,36000,0,14
4,appointment_date,datetime64[us],36000,0,729
5,appointment_weekday,int64,36000,0,7
6,appointment_hour,int64,36000,0,13
7,lead_days,int64,36000,0,161
8,visit_type,str,36000,0,11
9,specialty_group,str,36000,0,8


In [24]:
appointments.to_csv(
    '../data/clean/appointments_cleaned.csv',
    index=False
)